# Bước 0b: train lại probe cho từng loại đầu vào (cross-validation trên tập val)

Ở notebook oracle, probe của Meta **đóng băng** nên không đo được một predictor tốt hơn thật sự giúp bao nhiêu: probe đã quen với
đầu ra của predictor. Notebook này train **probe mới** cho từng loại đầu vào, trên token đã pool 4×4 theo từng bước thời gian:

| biến thể | đầu vào của probe |
|---|---|
| `enc` | chỉ token encoder (16 bước thời gian đã quan sát) |
| `enc_pred` | + token tương lai do **predictor V-JEPA 2** dự đoán (cách V-JEPA 2 làm) |
| `enc_real` | + token tương lai **thật** (giống một predictor hoàn hảo) |
| `enc_copylast` | + lặp lại bước cuối (đối chứng: thêm một ô nhưng không thêm thông tin mới) |
| `pred`, `real` | chỉ token tương lai |

Người tham gia của tập val được chia thành 2 fold, mỗi người chỉ nằm trong một fold. Mỗi clip được dự đoán bởi probe **không thấy** người đó
khi train, sau đó tính mean-class R@5 trên toàn tập val. Chạy 3 seed, báo cáo mean ± std, kèm khoảng tin cậy bootstrap.

**Cách đọc kết quả:**
- `enc_real − enc_pred` là khoảng còn có thể cải thiện nếu có predictor tốt hơn. Nếu lớn (vài điểm), nên pretrain lại predictor.
- `enc_pred − enc` là phần predictor hiện tại đóng góp.

Số tuyệt đối thấp hơn probe của Meta (dòng `released*`), vì ở đây chỉ train trên khoảng 4.6k clip và token đã bị pool. Chỉ nên so sánh
các biến thể với nhau.

**Cài đặt:** GPU T4 x2, Internet On, *Add Input* 2 dataset `ek100-292-val-0` và `ek100-292-val-1`. Notebook chạy khoảng 2–2.5 giờ:
trích token khoảng 1.5 giờ, train probe khoảng 30–60 phút. Output khoảng 6 GB. Nếu bị ngắt giữa chừng, *Add Input* output của lần chạy trước
rồi chạy lại: những clip đã trích token sẽ được bỏ qua.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"  # version the code was checked against
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq install -y aria2 > /dev/null 2>&1 && aria2c --version | head -1
!apt-get -qq install -y aria2 > /dev/null 2>&1 && pip install -q av

In [ ]:
# Checkpoints (~1.3GB encoder+predictor, probe) -> /tmp so they do not count toward the 20GB output limit
!mkdir -p /tmp/ckpt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/vitl.pt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitl-256.pt
!ls -lh /tmp/ckpt

In [ ]:
# Official annotations (the full train csv is required: probe output classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!ls /kaggle/input; find /kaggle/input -maxdepth 3 | head -30

In [ ]:
import glob, os, shutil
ANCHOR = "action_start"   # standard EK100 protocol ("official" also works: another ~1.5 h and ~6 GB)
OUT = "/kaggle/working/logits_feats"
prev = glob.glob("/kaggle/input/**/logits_feats", recursive=True)
if prev:
    shutil.copytree(prev[0], OUT, dirs_exist_ok=True); print("resumed from", prev[0])
mp4 = glob.glob("/kaggle/input/**/ek100_292/**/*.MP4", recursive=True)
assert len(mp4) >= 138, f"found {len(mp4)} videos: Add Input the datasets ek100-292-val-0 and ek100-292-val-1"
VROOT = os.path.commonpath([os.path.dirname(os.path.dirname(os.path.dirname(p))) for p in mp4])
print(f"using {len(mp4)} re-encoded videos under {VROOT}")
!python $R/scripts/prepare_ek100.py --video_root $VROOT \
    --train_csv /kaggle/working/ek100-ann/EPIC_100_train.csv \
    --val_csv /kaggle/working/ek100-ann/EPIC_100_validation.csv \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv --out_dir /kaggle/working/work | grep -E '"val_clips"|val_videos_found'


In [ ]:
# Encoder / predictor / real future tokens pooled to 4x4 per time step, one process per GPU (resumable)
import subprocess, time
def shard_cmd(gpu):
    run = (f"python {R}/scripts/eval_ek100.py --work_dir /kaggle/working/work --out_dir {OUT} "
           f"--vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt --anchors {ANCHOR} "
           f"--device cuda:{gpu} --shard_id {gpu} --num_shards 2 --dtype fp16 --num_workers 2 "
           f"--oracle --fix_frame_ids --save_feats_pool 4")
    return f"for i in 1 2 3 4 5; do {run} && break; echo \"[retry $i]\"; done"
procs = {g: subprocess.Popen(["bash", "-c", shard_cmd(g) + f" >> /kaggle/working/feats_gpu{g}.log 2>&1"]) for g in (0, 1)}
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for g in procs:
        lines = [l for l in open(f"/kaggle/working/feats_gpu{g}.log").read().splitlines()
                 if any(k in l for k in ("clip/s", "to go", "[error]", "[retry", "Traceback"))]
        print(f"gpu{g}: {lines[-1] if lines else 'running...'}", flush=True)
print("exit codes:", {g: p.returncode for g, p in procs.items()})
!du -sh $OUT


In [ ]:
# Frozen-probe reference on the same clips, then the cross-validated probes (6 variants x 2 folds x 3 seeds)
!python $R/scripts/summarize_oracle.py --work_dir /kaggle/working/work --logits_root $OUT --anchor $ANCHOR \
    --out /kaggle/working/oracle_{ANCHOR}.json
!python $R/scripts/cv_probe.py --work_dir /kaggle/working/work --logits_root $OUT --anchor $ANCHOR \
    --out /kaggle/working/cv_probe_{ANCHOR}.json --device cuda:0


Gửi lại output của cell cuối (bảng oracle và bảng cross-validation), kèm file `cv_probe_action_start.json`.